In [ ]:
import os, sys, subprocess, textwrap

game_code = textwrap.dedent('''
import tkinter as tk
from tkinter import messagebox
import threading
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

simulator = AerSimulator()

WINS = [
    (0,1,2),(3,4,5),(6,7,8),
    (0,3,6),(1,4,7),(2,5,8),
    (0,4,8),(2,4,6),
]

def measure_qubit(gate):
    qc = QuantumCircuit(1, 1)
    if gate == "x": qc.x(0)
    elif gate == "h": qc.h(0)
    qc.measure(0, 0)
    job = simulator.run(qc, shots=1, memory=True)
    return int(job.result().get_memory()[0])

class QuantumTicTacToe:
    def __init__(self, root):
        self.root = root
        self.root.title("Quantum Tic-Tac-Toe")
        self.root.resizable(False, False)
        self.root.configure(bg="#1a1a2e")
        self.board = [" "] * 9
        self.turn = 0
        self.busy = False
        self.buttons = []
        self.build_ui()

    def build_ui(self):
        tk.Label(self.root, text="Quantum Tic-Tac-Toe",
            font=("Helvetica", 22, "bold"),
            bg="#1a1a2e", fg="#e0e0ff", pady=14
        ).grid(row=0, column=0, columnspan=3)

        self.status = tk.Label(self.root,
            text="X turn - classical player",
            font=("Helvetica", 13),
            bg="#1a1a2e", fg="#aaaaff", pady=4)
        self.status.grid(row=1, column=0, columnspan=3)

        for i in range(9):
            btn = tk.Button(self.root,
                text=" ", font=("Helvetica", 36, "bold"),
                width=3, height=1,
                bg="#16213e", fg="#e0e0ff",
                activebackground="#0f3460",
                relief="flat", bd=0,
                highlightbackground="#444466",
                highlightthickness=2,
                command=lambda idx=i: self.on_click(idx))
            btn.grid(row=(i//3)+2, column=i%3, padx=8, pady=8)
            self.buttons.append(btn)

        self.log = tk.Text(self.root,
            height=7, width=42,
            font=("Courier", 11),
            state="disabled",
            bg="#0d0d1a", fg="#00ffaa",
            relief="flat", padx=8, pady=6)
        self.log.grid(row=5, column=0, columnspan=3, padx=12, pady=(6,4))

        tk.Button(self.root, text="Restart",
            font=("Helvetica", 12, "bold"),
            bg="#0f3460", fg="white",
            activebackground="#1a5276",
            relief="flat", padx=14, pady=6,
            command=self.restart
        ).grid(row=6, column=0, columnspan=3, pady=(4,16))

        self.log_message("Game started! X goes first.")
        self.log_message("X = classical  |  O = quantum (50/50)")

    def log_message(self, msg):
        self.log.config(state="normal")
        self.log.insert("end", f"  {msg}\n")
        self.log.see("end")
        self.log.config(state="disabled")

    def on_click(self, idx):
        if self.board[idx] != " " or self.busy:
            return
        self.busy = True
        self.disable_all()
        player = "X" if self.turn == 0 else "O"
        threading.Thread(target=self.process_move, args=(idx, player), daemon=True).start()

    def process_move(self, idx, player):
        if player == "X":
            result = measure_qubit("x")
        else:
            self.root.after(0, self.log_message, f"O plays cell {idx+1} - Hadamard - superposition...")
            result = measure_qubit("h")
        self.root.after(0, self.apply_move, idx, player, result)

    def apply_move(self, idx, player, result):
        if player == "X":
            self.board[idx] = "X"
            self.buttons[idx].config(text="X", fg="#ff6b6b", bg="#2d1b1b")
            self.log_message(f"X plays cell {idx+1} - claimed!")
        else:
            if result == 1:
                self.board[idx] = "O"
                self.buttons[idx].config(text="O", fg="#74b9ff", bg="#1b2d3d")
                self.log_message(f"  Collapsed 1 - O claims cell {idx+1}!")
            else:
                self.log_message(f"  Collapsed 0 - Quantum void! O loses turn.")

        winner = self.check_winner()
        if winner:
            self.log_message(f"{winner} wins!")
            self.status.config(text=f"{winner} wins!", fg="#00ff99")
            messagebox.showinfo("Game Over", f"{winner} wins!")
            self.busy = False
            return

        if all(c != " " for c in self.board):
            self.log_message("Its a draw!")
            self.status.config(text="Its a draw!", fg="#bf7fff")
            self.busy = False
            return

        self.turn = 1 - self.turn
        self.busy = False
        self.enable_empty()
        if self.turn == 0:
            self.status.config(text="X turn - classical player", fg="#ff9999")
        else:
            self.status.config(text="O turn - quantum player (50/50!)", fg="#74b9ff")

    def check_winner(self):
        for a,b,c in WINS:
            if self.board[a]==self.board[b]==self.board[c] and self.board[a]!=" ":
                for i in (a,b,c):
                    self.buttons[i].config(bg="#1a4731")
                self.disable_all()
                return self.board[a]
        return None

    def disable_all(self):
        for btn in self.buttons:
            btn.config(state="disabled")

    def enable_empty(self):
        for i, btn in enumerate(self.buttons):
            if self.board[i] == " ":
                btn.config(state="normal")

    def restart(self):
        self.board = [" "] * 9
        self.turn = 0
        self.busy = False
        for btn in self.buttons:
            btn.config(text=" ", state="normal", bg="#16213e", fg="#e0e0ff")
        self.status.config(text="X turn - classical player", fg="#aaaaff")
        self.log.config(state="normal")
        self.log.delete("1.0", "end")
        self.log.config(state="disabled")
        self.log_message("Game restarted! X goes first.")
        self.log_message("X = classical  |  O = quantum (50/50)")

if __name__ == "__main__":
    root = tk.Tk()
    root.tk.call("tk", "scaling", 2.0)
    app = QuantumTicTacToe(root)
    root.mainloop()
''')

script_path = os.path.join(os.getcwd(), '_qttt_game.py')
with open(script_path, 'w') as f:
    f.write(game_code)

subprocess.Popen([sys.executable, script_path])
print("Game window launched!")